In [4]:
%cd /run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough

# Standard library imports
import argparse
import importlib.util
import inspect
import json
import math
import os
import pickle
import random
import shutil
import socket
import subprocess
import sys
import tempfile
import warnings
from functools import reduce

# Third-party imports
import librosa
import lightning as L
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers.tensorboard import TensorBoardLogger
from matplotlib import cm
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
from tqdm import tqdm
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix
import umap
from pathlib import Path
import cv2
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold, train_test_split, StratifiedGroupKFold

from pytorch_grad_cam import GradCAM, HiResCAM, ScoreCAM, GradCAMPlusPlus, AblationCAM, XGradCAM, EigenCAM, FullGrad
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget, BinaryClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# Local imports
import commons
import lightning_wrapper
import models
import utils
import train
from cough_datasets import (
    CoughDatasets,
    CoughDatasetsCollate,
    CoughDatasetsProcessorCollate,
    CoughDetectionRatioBatchSampler,
    CoughDiseaseBinaryBatchSampler,
    PatientBatchSampler
)

/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough


In [5]:
now_experiment = "logs_ablation/b+mm_logmel"

experiment_metadata = now_experiment.split("/")
parser = train.parse_args()
args = parser.parse_args(["--model_name", experiment_metadata[1]])
model_dir = os.path.join(experiment_metadata[0], args.model_name)

config_path = args.config_path if args.init else os.path.join(model_dir, "config.json")
hps = train.load_config(config_path, model_dir, args)

# =============================================================
# SECTION: Loading Data
# =============================================================
df_train, df_test = train.load_data(hps)
collate_fn = train.get_collate_fn(hps)
target_labels = df_train[hps.data.target_column]

logger = utils.get_logger(hps.model_dir, filename="dummy.log")
logger.info(hps)

pool_net, pool_model = train.setup_model(hps, is_init=args.init)
runner_lightning = lightning_wrapper.CoughClassificationRunner.load_from_checkpoint(
    os.path.join(hps.model_dir, "best_model.ckpt"),
    model=pool_model,
    hps=hps, custom_logger=logger
)
runner_lightning.eval()

info_fold_data = train.load_fold_info(hps.model_dir)
best_fold_idx = info_fold_data.get("best_fold_idx", 0)
runner_lightning.probs_threshold = info_fold_data['best_threshold']

train_dataset = CoughDatasets(
    df_train.values, 
    hps.data,
    wav_stats_path=f"{hps.model_dir}/wav_stats_fold_{best_fold_idx}.pickle", 
    train=False
)
test_dataset = CoughDatasets(
    df_test.values, 
    hps.data,
    wav_stats_path=f"{hps.model_dir}/wav_stats_fold_{best_fold_idx}.pickle", 
    train=False
)

# Create sampler
#sampler = train.create_sampler(train_fold, hps)

# Create dataloaders
train_loader = DataLoader(
    train_dataset, 
    num_workers=28, 
    #sampler=sampler, 
    batch_size=hps.train.batch_size,
    pin_memory=True, 
    collate_fn=collate_fn
)
test_loader = DataLoader(
    test_dataset, 
    num_workers=28, 
    shuffle=False, 
    batch_size=hps.train.batch_size,
    pin_memory=True, 
    collate_fn=collate_fn
)

INFO:b+mm_logmel:{'train': {'use_cuda': True, 'log_interval': 20, 'seed': 1234, 'epochs': 10000, 'learning_rate': 0.0001, 'weight_decay': 0.0001, 'betas': [0.8, 0.99], 'eps': 1e-09, 'lr_decay': 0.999875, 'batch_size': 128, 'loss_function': 'BCE', 'use_Kfold': True, 'mae_training': False, 'ssccl_training': False}, 'data': {'max_wav_value': False, 'mean_std_norm': True, 'per_band_norm': False, 'sampling_rate': 16000, 'filter_length': 2048, 'hop_length': 256, 'win_length': 512, 'n_mel_channels': 80, 'mel_fmin': 50.0, 'mel_fmax': 8000.0, 'saming_length': True, 'desired_length': 0.55, 'fade_samples_ratio': 16, 'pad_types': 'zero', 'rezize_size': [64, 256], 'tabular_feature': True, 'acoustic_feature': True, 'feature_type': 'logmel', 'delta_feature': False, 'deltadelta_feature': False, 'multimask_augment': True, 'multimask_prob': 0.5, 'tau': 0.1, 'nu': 0.1, 'num_masks': 4, 'augment_data': True, 'augment_prob': 0.5, 'augment_rawboost': True, 'add_noise': False, 'cough_detection': False, 'mix_a

INFO:b+mm_logmel:Trainable params: 5978977 | Total params: 5978977 | Trainable%: 100.00% | Size: 5.98M


In [37]:
kf = KFold(n_splits=5, random_state=42, shuffle=True)
fold_splits = list(kf.split(df_train, target_labels))

test_labels = []
test_probs = []
test_preds = []
test_ids = []

memory = np.zeros_like(target_labels)
identified = []
for fold, (train_idx, test_idx) in enumerate(fold_splits):
    trainval_fold = df_train.iloc[train_idx].reset_index(drop=True)
    test_fold = df_train.iloc[test_idx].reset_index(drop=True)

    test_dataset = CoughDatasets(
        test_fold.values, 
        hps.data,
        wav_stats_path=f"{hps.model_dir}/wav_stats_fold_1.pickle", 
        train=False
    )

    test_loader = DataLoader(
        test_dataset, 
        num_workers=28, 
        shuffle=False, 
        batch_size=hps.train.batch_size,
        pin_memory=True, 
        collate_fn=collate_fn
    )

    fold_test_labels = []
    fold_test_probs = []
    fold_test_preds = []
    with torch.no_grad():
        for idx, batch in tqdm(enumerate(test_loader), total=len(test_loader)):
            wavnames, audio, _, attention_masks, dse_ids, [patient_ids, _, _, _] = batch
            audio = audio.cuda()
            attention_masks = attention_masks.cuda()
            out_model = runner_lightning.model.forward(audio, attention_mask=attention_masks)
            logits = out_model['disease_logits']

            probs = torch.sigmoid(logits).squeeze(-1)  # [B]
            preds = (probs >= runner_lightning.probs_threshold).long().cpu().detach().numpy()
            labels = torch.argmax(dse_ids, dim=1).cpu().detach().numpy()

            fold_test_labels.extend(labels)
            fold_test_probs.extend(probs.cpu().detach().numpy())
            fold_test_preds.extend(preds)

    test_labels.append(fold_test_labels)
    test_probs.append(fold_test_probs)
    test_preds.append(fold_test_preds)
    test_ids.append(test_idx)


100%|██████████| 12/12 [00:01<00:00,  8.63it/s]


In [41]:
runner_lightning.model.device

AttributeError: 'ResNet34ManualClassifier' object has no attribute 'device'

In [10]:
test_labels_all = np.concatenate(test_labels)
pred_probs_all = np.concatenate(test_probs)
test_ids_all = np.concatenate(test_ids)

In [ ]:
temp = np.where(test_labels_all == 1, pred_probs_all, 1 - pred_probs_all)
weights = 1 - (temp.max() - temp)
weights = weights[np.argsort(test_ids_all)]
memory = 0.3*weights + 0.7*memory

In [35]:
test_ids_all[np.argsort(test_ids_all)]

array([   0,    1,    2, ..., 7057, 7058, 7059])

In [33]:
test_ids_all

array([   8,   14,   17, ..., 7044, 7046, 7050])

In [32]:
np.argsort(test_ids_all)

array([2824, 4236, 4237, ..., 5647, 1411, 4235])

In [31]:
weights

array([0.9232969 , 0.94946444, 0.6362928 , ..., 0.77824867, 0.8025828 ,
       0.7528444 ], dtype=float32)

In [27]:
pred_probs_all

array([0.34149536, 0.34204614, 0.34389836, ..., 0.48388973, 0.62205416,
       0.5179219 ], dtype=float32)

In [ ]:
experiments = ["logs_ablation/b+mm_logmel"]
use_cpu = False

dfs = []
for now_experiment in experiments:


    # =============================================================
    # SECTION: Loading Data
    # =============================================================
    df_train, df_test = train.load_data(hps)
    collate_fn = train.get_collate_fn(hps)
    target_labels = df_train[hps.data.target_column]

    logger = utils.get_logger(hps.model_dir, filename="dummy.log")
    logger.info(hps)

    pool_net, pool_model = train.setup_model(hps, is_init=args.init)
    runner_lightning = lightning_wrapper.CoughClassificationRunner.load_from_checkpoint(
        os.path.join(hps.model_dir, "best_model.ckpt"),
        model=pool_model,
        hps=hps, custom_logger=logger
    )
    runner_lightning.eval()
    trainer = L.Trainer(accelerator="gpu" if use_cpu == False else "cpu", devices="auto")

    # with open(os.path.join(model_dir, "probs_threshold.pkl"), "rb") as f:
    #     runner_lightning.probs_threshold = pickle.load(f)['probs_threshold']

    info_fold_data = train.load_fold_info(hps.model_dir)
    best_fold_idx = info_fold_data.get("best_fold_idx", 0)
    runner_lightning.probs_threshold = info_fold_data['best_threshold']

    train_dataset = CoughDatasets(
        df_train.values, 
        hps.data,
        wav_stats_path=f"{hps.model_dir}/wav_stats_fold_{best_fold_idx}.pickle", 
        train=False
    )
    test_dataset = CoughDatasets(
        df_test.values, 
        hps.data,
        wav_stats_path=f"{hps.model_dir}/wav_stats_fold_{best_fold_idx}.pickle", 
        train=False
    )
    
    # Create sampler
    #sampler = train.create_sampler(train_fold, hps)
    
    # Create dataloaders
    train_loader = DataLoader(
        train_dataset, 
        num_workers=28, 
        #sampler=sampler, 
        batch_size=hps.train.batch_size,
        pin_memory=True, 
        collate_fn=collate_fn
    )
    test_loader = DataLoader(
        test_dataset, 
        num_workers=28, 
        shuffle=False, 
        batch_size=hps.train.batch_size,
        pin_memory=True, 
        collate_fn=collate_fn
    )

    test_wavnames = []
    test_probs = []
    test_preds = []
    test_labels = []
    #test_embeddings = []
    with torch.no_grad():
        for idx, batch in tqdm(enumerate(train_loader), total=len(train_loader)):
            wavnames, audio, _, attention_masks, dse_ids, [patient_ids, _, _, _] = batch
            audio = audio.cuda()
            attention_masks = attention_masks.cuda()
            out_model = runner_lightning.model.forward(audio, attention_mask=attention_masks)
            logits = out_model['disease_logits']

            probs = torch.sigmoid(logits).squeeze(-1)  # [B]
            preds = (probs >= runner_lightning.probs_threshold).long().cpu().detach().numpy()
            labels = torch.argmax(dse_ids, dim=1).cpu().detach().numpy()

            test_wavnames.extend(wavnames)
            test_labels.extend(labels)
            test_preds.extend(preds)
            test_probs.extend(probs.cpu().detach().numpy())
            #test_embeddings.extend(out_model['embeddings'].cpu().detach().numpy())

        for idx, batch in tqdm(enumerate(test_loader), total=len(test_loader)):
            wavnames, audio, _, attention_masks, dse_ids, [patient_ids, _, _, _] = batch
            audio = audio.cuda()
            attention_masks = attention_masks.cuda()
            out_model = runner_lightning.model.forward(audio, attention_mask=attention_masks)
            logits = out_model['disease_logits']

            probs = torch.sigmoid(logits).squeeze(-1)  # [B]
            preds = (probs >= runner_lightning.probs_threshold).long().cpu().detach().numpy()
            labels = torch.argmax(dse_ids, dim=1).cpu().detach().numpy()

            test_wavnames.extend(wavnames)
            test_labels.extend(labels)
            test_preds.extend(preds)
            test_probs.extend(probs.cpu().detach().numpy())
            #test_embeddings.extend(out_model['embeddings'].cpu().detach().numpy())

    del audio, attention_masks
    test_wavnames = np.array(test_wavnames)
    test_labels = np.array(test_labels)
    test_preds = np.array(test_preds)
    test_probs = np.array(test_probs)
    #test_embeddings = np.array(test_embeddings)

    df_now = pd.DataFrame({
        "path_file": test_wavnames,
        experiment_metadata[1]: test_preds,
        experiment_metadata[1] + "_probs": test_probs
    })
    dfs.append(df_now)

final_df = reduce(
    lambda left, right: pd.merge(left, right, on="path_file", how="inner"),
    dfs
)

df_train['split'] = "train"
df_test['split'] = "test"

df_all = pd.concat([df_train, df_test])
df_all.reset_index(inplace=True, drop=True)

df_result = df_all.merge(
    final_df,
    on="path_file",
    how="inner",
    validate="one_to_one"
)

In [3]:
parser = train.parse_args()
args = parser.parse_args(["--init", "--model_name", "dev", "--pooling_model", "PEFTWavLM_Try1", "--feature_dim", "1024", "--config_path", "configs/general.json"])
model_dir = os.path.join("./logs", args.model_name)
os.makedirs(model_dir, exist_ok=True)
port = None

config_path = args.config_path if args.init else os.path.join(model_dir, "config.json")
hps = train.load_config(config_path, model_dir, args)

df_train, _ = train.load_data(hps)
collate_fn = train.get_collate_fn(hps)
target_labels = df_train[hps.data.target_column]